In [47]:
import pandas as pd
import numpy as np

#Load dataset
df = pd.read_csv('FIFAWC2026.csv')

#Check some rows of dataset
print(df.head())



   Number            Player    Pos                   Club  \
0      15        Ãlex Baena  MF,FW  1.es AtlÃ©tico Madrid   
1       6      Mikel Merino     MF          1.eng Arsenal   
2      21   Mikel Oyarzabal     FW     1.es Real Sociedad   
3      12       Pedro Porro     DF        1.eng Tottenham   
4       8  Fabián Ruiz Peña     MF               1.fr PSG   

                         Birth Place Birth Date Age  MP  Min  Gls  Height  \
0             Roquetas de Mar, Spain  7/20/2001  24   7  481    1     175   
1                    Pamplona, Spain  6/22/1996  29   8  199    2     188   
2                       Eibar, Spain  4/21/1997  29   8  601    5     181   
3                  Don Benito, Spain  9/13/1999  26   6  563    2     176   
4  Los Palacios y Villafranca, Spain   4/3/1996  30   8  321    1     189   

   Weight   Team Team_Status  
0      69  Spain    advanced  
1      83  Spain    advanced  
2      78  Spain    advanced  
3      68  Spain    advanced  
4      69  Spai

In [48]:
#Remove unnecessary column from the dataset for my analysis
df = df.loc[:, ~df.columns.str.contains("Number|Club|Birth Place|Birth Date|Age")]

#Check remaining columns
print(df.columns)


Index(['Player', 'Pos', 'MP', 'Min', 'Gls', 'Height', 'Weight', 'Team',
       'Team_Status'],
      dtype='object')


In [49]:
#Checking rows after removing the column
print(df.head())

#Checking how many rows I have
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

             Player    Pos  MP  Min  Gls  Height  Weight   Team Team_Status
0        Ãlex Baena  MF,FW   7  481    1     175      69  Spain    advanced
1      Mikel Merino     MF   8  199    2     188      83  Spain    advanced
2   Mikel Oyarzabal     FW   8  601    5     181      78  Spain    advanced
3       Pedro Porro     DF   6  563    2     176      68  Spain    advanced
4  Fabián Ruiz Peña     MF   8  321    1     189      69  Spain    advanced
Rows: 1022
Columns: 9


In [57]:
#Map actual columns to standard working variables
column_mapping = {
        'Player': 'player_name',
        'Height': 'height_cm',
        'Weight': 'weight_kg',
        'Gls': 'goals',
        'Team_Status': 'team_status'
    }
#Rename column
existing_mapping = {k: v for k, v in column_mapping.items() if k in df.columns}
df_mapped = df.rename(columns=existing_mapping).copy()

print("=== Columns mapped successfully ===")
print(existing_mapping)

=== Columns mapped successfully ===
{'Player': 'player_name', 'Height': 'height_cm', 'Weight': 'weight_kg', 'Gls': 'goals', 'Team_Status': 'team_status'}


In [59]:
#Checking necessary columns are present after mapping
required_cols = ['height_cm', 'weight_kg', 'goals', 'team_status']
missing_cols = [col for col in required_cols if col not in df_mapped.columns]
if missing_cols:
        raise ValueError(f"Missing required columns in dataset: {missing_cols}. "
                         f"Please ensure columns 'Height', 'Weight', 'Gls' and 'Team_Status' exist.")

print("   ->", required_cols)
print(df_mapped[['player_name'] + required_cols].head())


   -> ['height_cm', 'weight_kg', 'goals', 'team_status']
        player_name  height_cm  weight_kg  goals team_status
0        Ãlex Baena        175         69      1    advanced
1      Mikel Merino        188         83      2    advanced
2   Mikel Oyarzabal        181         78      5    advanced
3       Pedro Porro        176         68      2    advanced
4  Fabián Ruiz Peña        189         69      1    advanced


In [60]:
#Check missing values
null_counts = df_mapped[required_cols].isnull().sum()
print("Detected missing values in raw columns:")
for col, count in null_counts.items():
   print(f"  - Column '{col}': {count} missing value(s)")

df_clean = df_mapped.dropna(subset=required_cols).copy()
print(f"Total rows after removing rows with missing values: {len(df_clean)}")

Detected missing values in raw columns:
  - Column 'height_cm': 0 missing value(s)
  - Column 'weight_kg': 0 missing value(s)
  - Column 'goals': 0 missing value(s)
  - Column 'team_status': 0 missing value(s)
Total rows after removing rows with missing values: 1022


In [61]:
# Filter out height and weight unrealistic values (e.g., heights < 140cm or weights < 40kg)
df_clean = df_clean[(df_clean['height_cm'] >= 140) & (df_clean['weight_kg'] >= 40)]

# Formula: BMI = Weight (kg) / [Height (m)]^2
df_clean['BMI'] = df_clean['weight_kg'] / ((df_clean['height_cm'] / 100) ** 2)

#Fix Team Status text so they all match
df_clean['team_status'] = df_clean['team_status'].astype(str).str.strip().str.lower()

#Filter 2 groups, just what I would like to compare
df_clean = df_clean[df_clean['team_status'].isin(['advanced', 'eliminated'])]

n_advanced_pop = len(df_clean[df_clean['team_status'] == 'advanced'])
n_eliminated_pop = len(df_clean[df_clean['team_status'] == 'eliminated'])

print(f"Total cleaned population size: {len(df_clean)} players")
print(f"  - Advanced group (32 teams): {n_advanced_pop} players")
print(f"  - Eliminated group (16 teams): {n_eliminated_pop} players")
print("--------------------------------------------------")


Total cleaned population size: 1020 players
  - Advanced group (32 teams): 707 players
  - Eliminated group (16 teams): 313 players
--------------------------------------------------


In [63]:
#Saving my cleaned dataset as a new CSV file
df_clean.to_csv("FIFAWC2026_BMI_Cleaned.csv",
                index=False
                )
print("Cleaned CSV file saved successfully.")


Cleaned CSV file saved successfully.
